# Lab 8.3 &mdash; Data Boundaries: Prompt, Trace, Vector Store

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 8 &mdash; Safety &amp; Guardrails**

### What you'll do
- Redact on the way in, using the allow-list you already wrote in Module 4
- Prove the trace you built yesterday holds no customer data
- Check what actually went into the vector index &mdash; the store you cannot easily un-write
- Decide retention, because not deciding is also a decision

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Three data stores, and you planned one of them.** The trace and the index are the
> ones that turn up in a review, and neither has anything to do with the model.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-8-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

## Concept

Everyone thinks about what goes into the prompt. Two other stores fill up quietly:

- the **trace**, which keeps every prompt and tool result, searchable, for as long as retention says
- the **vector store**, which keeps whatever was ingested, in chunks, and is awkward to un-write

The fix is the same in all three places and you have already written it: an **allow-list**, applied
on the way in.

## Section 1 &mdash; Redact on the way in

A record straight out of a ledger has more in it than the agent needs. The fields it does not need
are the fields that must never reach any of the three stores.

In [ ]:
RAW_RECORD = {
    "ref": "PMT-1003",
    "amount": 990000.00,
    "ccy": "USD",
    "counterparty": "ZENITH",
    "status": "held",
    "reason_code": "LIMIT_BREACH",
    # everything below here is real customer data the agent has no use for
    "beneficiary_name": "A. Sharma",
    "beneficiary_iban": "GB29NWBK60161331926819",
    "originator_account": "0021447788",
    "contact_email": "a.sharma@example.com",
    "contact_phone": "+44 7700 900123",
    "internal_memo": "client called, very unhappy",
}

AGENT_FIELDS = ("ref", "amount", "ccy", "counterparty", "status", "reason_code")

PII_FIELDS = ("beneficiary_name", "beneficiary_iban", "originator_account",
              "contact_email", "contact_phone", "internal_memo")

def redact(record: dict, allow=AGENT_FIELDS) -> dict:
    """Keep only what the agent needs. An allow-list, not a block-list."""
    return {k: v for k, v in record.items() if k in allow}


def leaks(payload) -> list:
    """Which PII values appear anywhere in this payload, serialised."""
    blob = json.dumps(payload, default=str).lower()
    return [f for f in PII_FIELDS
            if str(RAW_RECORD[f]).lower() in blob]

In [ ]:
# --- Self-check: Section 1
check("the raw record leaks every PII field",
      lambda: len(leaks(RAW_RECORD)) == len(PII_FIELDS))
check("the redacted record leaks none",
      lambda: leaks(redact(RAW_RECORD)) == [])
check("and still carries everything the agent needs",
      lambda: set(redact(RAW_RECORD)) == set(AGENT_FIELDS))
check("a field added to the source next year is dropped without anyone updating a list",
      lambda: leaks(redact({**RAW_RECORD, "passport_no": "X1234567"})) == []
              and "passport_no" not in redact({**RAW_RECORD, "passport_no": "X1234567"}),
      "this is the property a block-list does not have")
check("redaction is not lossy for the decision",
      lambda: redact(RAW_RECORD)["reason_code"] == "LIMIT_BREACH")

guard(lambda: print("  agent sees:", json.dumps(redact(RAW_RECORD))))

## Section 2 &mdash; The trace is a data store

Module 7's tracer recorded inputs and outputs. Point the same test at it: whatever you write into
a span is persisted, searchable, and outlives the run.

In [ ]:
TRACE = []          # stands in for Module 7's span store

def span(name: str, payload: dict, redacted: bool = True):
    """Record one span. What you put in here is what the trace store keeps."""
    TRACE.append({"name": name, "payload": redact(payload) if redacted else payload})


def trace_leaks() -> list:
    """Which PII fields are sitting in the trace right now."""
    return leaks(TRACE)


def run_and_trace(redacted: bool = True):
    TRACE.clear()
    span("ledger.lookup", RAW_RECORD, redacted=redacted)
    span("policy.decide", {"reason_code": RAW_RECORD["reason_code"]}, redacted=redacted)
    return trace_leaks()

In [ ]:
# --- Self-check: Section 2
check("tracing the raw record puts every PII field in the trace store",
      lambda: len(run_and_trace(redacted=False)) == len(PII_FIELDS),
      "an observability improvement, and a copy of the customer database")
check("tracing the redacted record leaks nothing",
      lambda: run_and_trace(redacted=True) == [])
check("the trace still records what happened",
      lambda: (run_and_trace(True), len(TRACE))[1] == 2)
check("and still identifies the case",
      lambda: (run_and_trace(True), TRACE[0]["payload"]["ref"])[1] == "PMT-1003",
      "you can debug from a redacted trace; you cannot un-write an unredacted one")
check("this is the SAME allow-list, pointed somewhere else",
      lambda: (run_and_trace(True), TRACE[0]["payload"] == redact(RAW_RECORD))[1] is True)

def _traces():
    for red in (False, True):
        found = run_and_trace(redacted=red)
        print(f"  redacted={str(red):5} -> {len(found)} PII field(s) in the trace {found[:3]}")
guard(_traces)

## Section 3 &mdash; The index you cannot un-write

A trace expires. An embedded chunk sits in the index until somebody re-indexes, and is retrievable
by everyone the retriever serves. Check what went in *before* it goes in.

In [ ]:
DOCS_TO_INDEX = [
    {"source": "runbook-v4.md", "text": "Payments above USD 500,000 require Treasury approval."},
    {"source": "runbook-v4.md", "text": "A payment held for SANCTIONS_REVIEW is decided by Compliance."},
    # somebody exported a case file into the knowledge base
    {"source": "case-notes.md",
     "text": "PMT-1003 beneficiary A. Sharma, IBAN GB29NWBK60161331926819, called and was unhappy."},
]

INDEXABLE_SOURCES = {"runbook-v4.md", "policy-v2.md"}

def safe_to_index(doc: dict) -> bool:
    """Two conditions, and both must hold before anything is embedded."""
    return doc["source"] in INDEXABLE_SOURCES and leaks(doc["text"]) == []


def index_report() -> dict:
    ok = [d for d in DOCS_TO_INDEX if safe_to_index(d)]
    return {"indexed": len(ok),
            "rejected": [d["source"] for d in DOCS_TO_INDEX if not safe_to_index(d)]}

In [ ]:
# --- Self-check: Section 3
check("the two runbook chunks are safe to index",
      lambda: index_report()["indexed"] == 2)
check("the exported case file is rejected",
      lambda: index_report()["rejected"] == ["case-notes.md"])
check("it would be rejected on its SOURCE alone",
      lambda: safe_to_index({"source": "case-notes.md", "text": "nothing sensitive here"})
              is False,
      "an allow-list of sources is the cheap check, and it runs before you read a word")
check("and on its CONTENT alone, even from an allowed source",
      lambda: safe_to_index({"source": "runbook-v4.md",
                             "text": "example: IBAN GB29NWBK60161331926819"}) is False,
      "belt and braces, because somebody will paste a real case into the runbook")
check("both conditions are required, not either",
      lambda: safe_to_index({"source": "runbook-v4.md", "text": "clean"}) is True)

def _index():
    r = index_report()
    print(f"  indexed {r['indexed']} of {len(DOCS_TO_INDEX)}; rejected {r['rejected']}")
    print("  A trace expires. This one does not -- deleting a chunk means re-indexing.")
guard(_index)

## Section 4 &mdash; Retention is a decision

Not setting it is also a decision, and it is the one that gets made by default.

In [ ]:
RETENTION_DAYS = {"prompt": 0, "trace": 30, "vector_store": None}   # None = forever

def retention_review() -> list:
    """One row per store: how long it keeps data, and whether that was chosen."""
    rows = []
    for store, days in RETENTION_DAYS.items():
        rows.append({"store": store,
                     "days": days,
                     "forever": days is None,
                     "decided": days is not None})
    return rows


def undecided() -> list:
    return [r["store"] for r in retention_review() if not r["decided"]]

In [ ]:
# --- Self-check: Section 4
check("every store is reviewed",
      lambda: len(retention_review()) == 3)
check("the vector store keeps data forever",
      lambda: any(r["forever"] for r in retention_review()))
check("and that is the one nobody decided",
      lambda: undecided() == ["vector_store"],
      "'forever' is what you get when the question is never asked")
check("the prompt keeps nothing, which is the only store that is safe by construction",
      lambda: RETENTION_DAYS["prompt"] == 0)
check("the trace has a number, so somebody chose it",
      lambda: RETENTION_DAYS["trace"] > 0)

## Run it for real

Send a redacted and an unredacted record to the model and ask each to recommend an action. The
question is whether the PII was ever load-bearing.

In [ ]:
if llm_ready():
    def _does_pii_help():
        for label, payload in (("redacted  ", redact(RAW_RECORD)), ("full record", RAW_RECORD)):
            reply = ask("You are a payments operations agent. Recommend one action for this case "
                        "in a single short sentence.\n\n" + json.dumps(payload, default=str))
            print(f"  [{label}] {reply.strip()[:160]}")
    guard(_does_pii_help)

### Read it

If the two recommendations are the same &mdash; and they should be, because the decision turns on
`status` and `reason_code` &mdash; then every PII field you sent was pure liability. It bought nothing
and it is now in the prompt, the trace, and anywhere else that context was copied.

That is the usual finding. The fields go in because the tool returned them and nobody filtered,
not because anything needed them.

**What you take from this lab:** redact where the data enters, not where it leaves; point the same
allow-list at the prompt, the trace and the index; and give every store a retention number that
somebody chose.

In [ ]:
score()

## Your turn

1. `leaks` matches exact values, which is the easy case. Real leakage is paraphrase &mdash;
   &ldquo;the Sharma payment&rdquo;. What would you actually have to check, and can you check it cheaply?
2. Your trace needs to be debuggable. Replace redaction with a stable pseudonym per beneficiary,
   so a support engineer can follow one customer across runs without seeing a name. What have you
   just created, and where does the mapping live?
3. Set a retention number for the vector store and write the sentence justifying it. If you cannot
   write the sentence, you have found the actual problem.